In [5]:
!pip -q install -U ultralytics roboflow pyyaml

In [6]:
import os
import cv2
import yaml
import shutil
import random
import zipfile
import numpy as np

from pathlib import Path
from collections import Counter

from ultralytics import YOLO
from roboflow import Roboflow

import matplotlib.pyplot as plt

In [7]:
import torch

print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU not enabled!")

Torch : 2.11.0+cu128
CUDA  : True
Tesla T4


In [8]:
rf = Roboflow(api_key="8MqIaual7OyR3dQXZMRj")
project = rf.workspace("apaar-mathur").project("safety-food-system-aecc8-znomq")
version = project.version(2)
dataset = version.download("yolov8")
HAIRNET = Path(dataset.location)

print(HAIRNET)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Safety-Food-System-2 in yolov8:: 100%|██████████| 24495/24495 [00:04<00:00, 6050.55it/s]


/content/Safety-Food-System-2


In [9]:
for x in HAIRNET.iterdir():
    print(x)

/content/Safety-Food-System-2/README.roboflow.txt
/content/Safety-Food-System-2/valid
/content/Safety-Food-System-2/README.dataset.txt
/content/Safety-Food-System-2/test
/content/Safety-Food-System-2/train
/content/Safety-Food-System-2/data.yaml


In [11]:
ZIP = "/content/merged_hygiene.zip"

DEST = Path("/content")

with zipfile.ZipFile(ZIP,"r") as z:
    z.extractall(DEST)

print("Done")

Done


In [12]:
HYGIENE_CLASSES = [
    "apron",          # 0
    "no_apron",       # 1
    "gloves",         # 2
    "no_gloves",      # 3
    "hairnet",        # 4
    "no_hairnet",     # 5
    "with_mask",      # 6
    "without_mask",   # 7
    "head",           # 8
]

In [13]:
NEW_DATASET = Path("/content/hairnet_dataset")

if NEW_DATASET.exists():
    shutil.rmtree(NEW_DATASET)

for split in ["train", "valid", "test"]:

    (NEW_DATASET/split/"images").mkdir(parents=True, exist_ok=True)
    (NEW_DATASET/split/"labels").mkdir(parents=True, exist_ok=True)

    img_dir = HAIRNET/split/"images"
    lbl_dir = HAIRNET/split/"labels"

    for img in img_dir.glob("*"):

        label = lbl_dir/f"{img.stem}.txt"

        if not label.exists():
            continue

        new_labels = []

        with open(label) as f:

            for line in f:

                cls,*box = line.strip().split()

                cls = int(cls)

                # ------------------------------
                if cls == 0:
                    cls = 8

                # hairnet
                elif cls == 1:
                    cls = 4

                # rats
                else:
                    continue

                new_labels.append(
                    f"{cls} {' '.join(box)}"
                )

        if not new_labels:
            continue

        shutil.copy2(img, NEW_DATASET/split/"images"/img.name)

        with open(
            NEW_DATASET/split/"labels"/label.name,
            "w"
        ) as f:
            f.write("\n".join(new_labels))

print("Conversion Complete")

Conversion Complete


In [14]:
MERGED = Path("/content/merged_hygiene_v2")

if MERGED.exists():
    shutil.rmtree(MERGED)

for split in ["train","valid","test"]:

    (MERGED/split/"images").mkdir(parents=True, exist_ok=True)
    (MERGED/split/"labels").mkdir(parents=True, exist_ok=True)

    # Existing merged dataset
    for img in (Path("/content/merged_hygiene")/split/"images").glob("*"):

        shutil.copy2(
            img,
            MERGED/split/"images"/img.name
        )

    for lbl in (Path("/content/merged_hygiene")/split/"labels").glob("*"):

        shutil.copy2(
            lbl,
            MERGED/split/"labels"/lbl.name
        )

    # Hairnet dataset
    for img in (NEW_DATASET/split/"images").glob("*"):

        shutil.copy2(
            img,
            MERGED/split/"images"/f"hairnet_{img.name}"
        )

    for lbl in (NEW_DATASET/split/"labels").glob("*"):

        shutil.copy2(
            lbl,
            MERGED/split/"labels"/f"hairnet_{lbl.name}"
        )

print("Merge Complete")

Merge Complete


In [15]:
cfg = {

    "path": str(MERGED),

    "train": "train/images",

    "val": "valid/images",

    "test": "test/images",

    "nc": 9,

    "names": HYGIENE_CLASSES
}

with open(MERGED/"data.yaml","w") as f:
    yaml.dump(cfg,f,sort_keys=False)

print("data.yaml created")

data.yaml created


In [16]:
counter = Counter()

for txt in MERGED.rglob("*.txt"):

    with open(txt) as f:

        for line in f:

            counter[int(line.split()[0])] += 1

print("="*80)

for i,name in enumerate(HYGIENE_CLASSES):
    print(f"{i:>2} : {name:20} {counter[i]:>8}")

print("="*80)

 0 : apron                       0
 1 : no_apron                    0
 2 : gloves                      0
 3 : no_gloves                   0
 4 : hairnet                 10029
 5 : no_hairnet                  0
 6 : with_mask                   0
 7 : without_mask                0
 8 : head                    10198


In [17]:
import shutil
from pathlib import Path

folder="/content/merged_hygiene_v2"

folder = Path(folder)
zip_name = f"/content/{folder.name}"
shutil.make_archive(zip_name, "zip", folder)
print(f"✅ Created: {zip_name}.zip")

✅ Created: /content/merged_hygiene_v2.zip


In [ ]:
model = YOLO("yolo11s.pt")

model.train(

    data=str(MERGED/"data.yaml"),

    epochs=120,

    imgsz=(640,1024),

    rect=True,

    batch=-1,

    cache=True,

    workers=4,

    optimizer="AdamW",

    lr0=1e-3,

    lrf=1e-2,

    cos_lr=True,

    weight_decay=5e-4,

    warmup_epochs=3,

    patience=20,

    close_mosaic=15,

    mosaic=0.5,

    mixup=0,

    copy_paste=0,

    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.05,
    scale=0.30,

    fliplr=0.5,

    amp=True,

    project="KitchenCam",

    name="Hygiene_YOLO11s",

    exist_ok=True
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/merged_hygiene_v2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=(640, 1024), iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=Hygiene_YOLO11s, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap

In [ ]:
best = YOLO(
    "/content/KitchenCam/Hygiene_YOLO11s/weights/best.pt"
)

metrics = best.val(
    data=str(MERGED/"data.yaml"),
    split="test",
    plots=True
)

print(metrics)